In [0]:
# Leer en Delta
DELTA_PATH = "/Volumes/workspace/default/network_data/delta/"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── CARGAR DESDE DELTA ────────────────────────────────────────────────────────
DELTA_PATH = "/Volumes/workspace/default/network_data/delta/"
df = spark.read.format("delta").load(DELTA_PATH)

print(f"Total registros: {df.count():,}")
print(f"Total columnas:  {len(df.columns)}")


In [0]:
DELTA_PATH = "/Volumes/workspace/default/network_data/delta/"
df = spark.read.format("delta").load(DELTA_PATH)

In [0]:
df.printSchema()
display(df.limit(10))
#

In [0]:
# ── VALORES FALTANTES POR COLUMNA ─────────────────────────────────────────────
nulos = df.agg(*[
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])

display(nulos)


In [0]:
# ── PORCENTAJE DE NULOS POR COLUMNA ───────────────────────────────────────────
total = df.count()

nulos_pct = df.agg(*[
    F.round(
        F.count(F.when(F.col(c).isNull(), c)) / total * 100, 2
    ).alias(c)
    for c in df.columns
])

display(nulos_pct)


In [0]:
# ── IMPUTACIÓN DE COLUMNAS MODBUS ─────────────────────────────────────────────

columnas_categoricas_modbus = [
    "appl_name",
    "proxy_src_ip",
    "Modbus_Function_Code",
    "Modbus_Function_Description",
    "Modbus_Transaction_ID",
    "SCADA_Tag",
    "Modbus_Value"
]

df_imputed = df.fillna("Non-Modbus", subset=columnas_categoricas_modbus)

# Verificación — deben ser 0 nulos en esas columnas
display(
    df_imputed.agg(*[
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in columnas_categoricas_modbus
    ])
)

In [0]:
display(df_imputed.describe())

In [0]:
# ── VALORES ÚNICOS POR COLUMNA CATEGÓRICA ─────────────────────────────────────
categoricas = [c for c in df.columns if c != "num"]

for col in categoricas:
    n = df.select(col).distinct().count()
    print(f"{col:35s} → {n:,} valores únicos")

# Crea una tabla con los valores unicos menores de 500 y muestra estos valores unicos junto con su distribución.
cols_menos_500 = [col for col in categoricas if df.select(col).distinct().count() <= 500]

for col in cols_menos_500:
    display(
        df.groupBy(col)
        .agg(F.count("*").alias("count"))
        .orderBy("count", ascending=False)
    )
#


In [0]:
# ── DISTRIBUCIÓN DE LA COLUMNA TAG ───────────────────────────────────────────
display(
    df_imputed.groupBy("Tag")
    .agg(F.count("*").alias("count"))
    .withColumn("porcentaje", F.round(F.col("count") / df.count() * 100, 2))
    .orderBy("count", ascending=False)
)


In [0]:
# ── GAPS TEMPORALES CON PARTICIÓN POR DÍA ────────────────────────────────────
from pyspark.sql.window import Window

w = Window.partitionBy("date").orderBy("timestamp")

df_gaps = df_imputed \
    .filter(F.col("timestamp").isNotNull()) \
    .withColumn("prev_timestamp", F.lag("timestamp").over(w)) \
    .withColumn("gap_seconds",
        F.unix_timestamp("timestamp") - F.unix_timestamp("prev_timestamp")
    ) \
    .filter(F.col("gap_seconds").isNotNull())


In [0]:
# ── ESTADÍSTICAS DE GAPS ──────────────────────────────────────────────────────
display(df_gaps.agg(
    F.min("gap_seconds").alias("gap_min_seg"),
    F.max("gap_seconds").alias("gap_max_seg"),
    F.mean("gap_seconds").alias("gap_medio_seg"),
    F.percentile_approx("gap_seconds", 0.5).alias("gap_mediana_seg"),
    F.percentile_approx("gap_seconds", 0.95).alias("gap_p95_seg"),
    F.percentile_approx("gap_seconds", 0.99).alias("gap_p99_seg"),
))


In [0]:
# ── GAPS GRANDES (ajusta el umbral según lo que veas arriba) ──────────────────
UMBRAL_SEGUNDOS = 60

display(
    df_gaps
    .filter(F.col("gap_seconds") > UMBRAL_SEGUNDOS)
    .select("prev_timestamp", "timestamp", "gap_seconds")
    .orderBy("gap_seconds", ascending=False)
)


In [0]:
# ── DISTRIBUCIÓN DE GAPS POR MAGNITUD ─────────────────────────────────────────
display(
    df_gaps.select(
        F.when(F.col("gap_seconds") == 0, "0 seg")
         .when(F.col("gap_seconds") <= 1, "1 seg")
         .when(F.col("gap_seconds") <= 10, "2-10 seg")
         .when(F.col("gap_seconds") <= 60, "11-60 seg")
         .when(F.col("gap_seconds") <= 3600, "1min-1h")
         .otherwise("> 1h")
         .alias("rango_gap")
    )
    .groupBy("rango_gap")
    .agg(F.count("*").alias("count"))
    .orderBy("count", ascending=False)
)


In [0]:
# ── GUARDAR EL DATAFRAME IMPUTADO EN DELTA ────────────────────────────────────
DELTA_PATH = "/Volumes/workspace/default/network_data/delta/"

df_imputed \
    .repartition(200, "date") \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("date") \
    .save(DELTA_PATH)

print("✅ Delta actualizado con imputación.")
